In [4]:
import pandas as pd
import numpy as np

# =========================================================
# 0. CARGA Y NORMALIZACIÓN
# =========================================================
df    = pd.read_csv('../data/hr_employees_processed.csv')
df_jh = pd.read_csv('../data/hr_job_history_processed.csv')

# Normalizar nombres de columnas: minúsculas y sin espacios
df.columns    = df.columns.str.strip().str.lower()
df_jh.columns = df_jh.columns.str.strip().str.lower()

# Conversión de fechas (ANTES de inspeccionar)
df['hiredate']      = pd.to_datetime(df['hiredate'])
df_jh['start_date'] = pd.to_datetime(df_jh['start_date'])
df_jh['end_date']   = pd.to_datetime(df_jh['end_date'])

print("=== 0. INSPECCIÓN GENERAL ===")
print(df.info())
print("\nNulos por columna:\n", df.isnull().sum())
print("\nTipos de fecha:")
print("  hiredate:  ", df['hiredate'].dtype)
print("  start_date:", df_jh['start_date'].dtype)
print("  end_date:  ", df_jh['end_date'].dtype)


# =========================================================
# 1. ANÁLISIS DE COMPENSACIÓN SALARIAL
# =========================================================
print("\n=== 1. COMPENSACIÓN SALARIAL ===")
print(df[['salary', 'commissionpct', 'annualcomp']].describe())

print("\nSalario medio por departamento:")
print(
    df.groupby('departmentname')['salary']
      .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
      .round(2)
      .sort_values('mean', ascending=False)
)

print("\nTop 5 puestos mejor pagados (salario medio):")
print(
    df.groupby('jobtitle')['salary'].mean().round(2)
      .sort_values(ascending=False).head(5)
)

if 'gender' in df.columns:
    print("\nBrecha salarial por género:")
    print(df.groupby('gender')['salary'].mean().round(2))


# =========================================================
# 2. DISTRIBUCIÓN DEMOGRÁFICA
# =========================================================
print("\n=== 2. DISTRIBUCIÓN DEMOGRÁFICA ===")

print("\nPor región:")
print(df.groupby('regionname')['employeeid'].nunique().sort_values(ascending=False))

print("\nPor país (top 10):")
print(
    df.groupby('countryname')['employeeid'].nunique()
      .sort_values(ascending=False).head(10)
)

print("\nPor ciudad (top 10):")
print(
    df.groupby('city')['employeeid'].nunique()
      .sort_values(ascending=False).head(10)
)

print("\nAntigüedad (años):")
print(df['tenureyears'].describe())


# =========================================================
# 3. ESTRUCTURA ORGANIZACIONAL
# =========================================================
print("\n=== 3. ESTRUCTURA ORGANIZACIONAL ===")

print("\nEmpleados por departamento (incluye NaN):")
print(
    df.groupby('departmentname', dropna=False)['employeeid'].count()
      .sort_values(ascending=False)
)

print("\nRatio de span of control (subordinados por manager):")
span = (
    df[df['managerid'].notna()]
      .groupby('managername')['employeeid']
      .count()
)
print(span.sort_values(ascending=False))

print("\nNiveles jerárquicos (raíz = sin manager):")
roots = df[df['managerid'].isna()]['employeename'].tolist()
print("Top-level managers:", roots)


# =========================================================
# 4. ROTACIÓN DE PERSONAL Y CAMBIOS DE TRABAJO
# =========================================================
print("\n=== 4. ROTACIÓN / CAMBIOS DE TRABAJO ===")
print(f"Total de cambios de puesto registrados: {len(df_jh)}")
print(f"Empleados con al menos 1 cambio: {df_jh['employee_id'].nunique()}")

print("\nCambios por departamento destino:")
print(
    df_jh.groupby('department_name')['employee_id'].count()
         .sort_values(ascending=False)
)

print("\nDuración media en cada puesto (días):")
df_jh['durationdays'] = (df_jh['end_date'] - df_jh['start_date']).dt.days
print(df_jh['durationdays'].describe())


# =========================================================
# 5. ANÁLISIS DE DESEMPEÑO POR DEPARTAMENTO
# =========================================================
print("\n=== 5. DESEMPEÑO POR DEPARTAMENTO (proxy) ===")
dept_kpi = (
    df.groupby('departmentname', dropna=False)
      .agg(
          empleados=('employeeid', 'count'),
          salariomedio=('salary', 'mean'),
          salariomax=('salary', 'max'),
          antiguedadmedia=('tenureyears', 'mean'),
          con_comision=('commissionpct', lambda s: (s > 0).sum()),
      )
      .round(2)
      .sort_values('salariomedio', ascending=False)
)
print(dept_kpi)

# Exportar resultados (opcional)
dept_kpi.to_csv('../data/dept_kpi.csv', index=True)
print("\n✅ Resultados exportados a '../data/dept_kpi.csv'")

=== 0. INSPECCIÓN GENERAL ===
<class 'pandas.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   employeeid        110 non-null    int64         
 1   firstname         110 non-null    str           
 2   lastname          110 non-null    str           
 3   employeename      110 non-null    str           
 4   email             110 non-null    str           
 5   phonenumber       110 non-null    str           
 6   hiredate          110 non-null    datetime64[us]
 7   jobid             110 non-null    str           
 8   jobtitle          110 non-null    str           
 9   minsalary         110 non-null    int64         
 10  maxsalary         110 non-null    int64         
 11  salary            110 non-null    float64       
 12  commissionpct     110 non-null    float64       
 13  annualcomp        110 non-null    int64         
 14  manager